[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/navjotts/ML-experiments/blob/master/15%20-%20Applying%20Logistic%20Regression/Applying_Logistic_Regression.ipynb)

# Target

1.   Use Logistic Regression to identify survivors of the titanic dataset
2.   Try converting the categorical features in titanic into OneHot features. Can we improve results? [One Hot Features](https://yashuseth.blog/2017/12/14/how-to-one-hot-encode-categorical-variables-of-a-large-dataset-in-python/)
3.   Try creating a few "feature cross" columns, and see if that improves results [Feature Crosses](https://developers.google.com/machine-learning/crash-course/feature-crosses/video-lecture)



### Data prep

In [1]:
import numpy as np
import pandas as pd
pd.options.display.max_rows = 10
from sklearn.preprocessing import LabelEncoder
import seaborn as sns

titanic = sns.load_dataset('titanic')

# drop duplicate/analogous columns
titanic = titanic.drop(['alive',
                        'adult_male',
                        'sex',
                        'class',
                        'embark_town'], axis=1)

# take care of missing data
titanic['embarked'] = titanic['embarked'].fillna(method='ffill')
titanic = titanic.drop(['deck'], axis=1)
titanic['age'] = titanic['age'].fillna(method='ffill')

# convert binomials and categoricals to encoded labels
for label in ['embarked', 'who', 'alone']:
    titanic[label] = LabelEncoder().fit_transform(titanic[label])

titanic.head()

,survived,pclass,age,sibsp,parch,fare,embarked,who,alone
0,0,3,22.0,1,0,7.2500,2,1,0
1,1,1,38.0,1,0,71.2833,0,2,0
2,1,3,26.0,0,0,7.9250,2,2,1
3,1,1,35.0,1,0,53.1000,2,2,0
4,0,3,35.0,0,0,8.0500,2,1,1


### Features and Labels

In [2]:
features_dataset = titanic.drop(['survived'], axis=1)
features = features_dataset.columns.values.tolist()
features

['pclass', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'who', 'alone']

In [3]:
labels = 'survived'
labels

'survived'

### Logistic Regression predictions on the `training` set

In [0]:
from sklearn import linear_model

# returns y_predict and the error
def run_Logistic_Regression_on_training(x, y, logging=True):
  print("Training starts...")
  regr = linear_model.LogisticRegression()
  regr.fit(x, y)
  print("Training ends...")
  print("Predicting...")
  y_predict = regr.predict(x)
  error = y_predict-y
  
  if logging:
    print("--------------------------------------------------------------------------------")
    print("| Number of misclassifications: %d (out of %d)" % (sum(error!=0), len(error)))
    print("| Error: ", sum(error!=0)/len(error))
    print("--------------------------------------------------------------------------------")  
  
  return y_predict, error

In [5]:
predicted, error = run_Logistic_Regression_on_training(titanic[features].as_matrix(), 
                                                       titanic[labels].as_matrix())

Training starts...
Training ends...
Predicting...
--------------------------------------------------------------------------------
| Number of misclassifications: 221 (out of 891)
| Error:  0.24803591470258138
--------------------------------------------------------------------------------


### Logistic Regression predictions on the `test` set

In [0]:
from sklearn import linear_model
from sklearn.model_selection import train_test_split

# returns y_predict and the error
def run_Logistic_Regression(x, y, logging=True):
  print("Splitting dataset...")
  x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
  print("Training set size: ", x_train.shape)
  print("Test set size: ", x_test.shape)
  
  print("Training starts...")
  regr = linear_model.LogisticRegression()
  regr.fit(x_train, y_train)
  print("Training ends...")
  print("Predicting...")
  y_predict = regr.predict(x_test)
  error = y_predict-y_test
  
  if logging:
    print("--------------------------------------------------------------------------------")
    print("| Number of misclassifications: %d (out of %d)" % (sum(error!=0), len(error)))
    print("| Error: ", sum(error!=0)/len(error))
    print("--------------------------------------------------------------------------------")  
  
  return y_predict, error

In [7]:
predicted, error = run_Logistic_Regression(titanic[features].as_matrix(), 
                                           titanic[labels].as_matrix())

Splitting dataset...
Training set size:  (712, 8)
Test set size:  (179, 8)
Training starts...
Training ends...
Predicting...
--------------------------------------------------------------------------------
| Number of misclassifications: 42 (out of 179)
| Error:  0.2346368715083799
--------------------------------------------------------------------------------


### Improve accuracy – 1) OneHot encodings

lets add OneHot Feature for `who`

In [0]:
# returns the updated DataFrame
def add_one_hot_feature(data, features):
  for feature in features:    
    one_hot_feature = pd.get_dummies(data[feature], prefix=feature)
    data = data.drop([feature], axis=1)
    data = pd.concat([data, one_hot_feature.iloc[:, 1:]], axis=1) # drop the 1st dummy_feature to avoid dummy variable trap    
  return data

In [9]:
titanic_updated = add_one_hot_feature(titanic, ['who'])
titanic_updated.head()

,survived,pclass,age,sibsp,parch,fare,embarked,alone,who_1,who_2
0,0,3,22.0,1,0,7.2500,2,0,1,0
1,1,1,38.0,1,0,71.2833,0,0,0,1
2,1,3,26.0,0,0,7.9250,2,1,0,1
3,1,1,35.0,1,0,53.1000,2,0,0,1
4,0,3,35.0,0,0,8.0500,2,1,1,0


In [10]:
features_dataset = titanic_updated.drop(['survived'], axis=1)
features = features_dataset.columns.values.tolist()
features

['pclass',
 'age',
 'sibsp',
 'parch',
 'fare',
 'embarked',
 'alone',
 'who_1',
 'who_2']

In [11]:
predicted, error = run_Logistic_Regression(titanic_updated[features].as_matrix(), 
                                           titanic_updated[labels].as_matrix())

Splitting dataset...
Training set size:  (712, 9)
Test set size:  (179, 9)
Training starts...
Training ends...
Predicting...
--------------------------------------------------------------------------------
| Number of misclassifications: 35 (out of 179)
| Error:  0.19553072625698323
--------------------------------------------------------------------------------


**Observation:** improvement in accuracy, error went down from 0.23 to 0.19

lets try OneHot features for  `pclass`

In [12]:
titanic_updated = add_one_hot_feature(titanic, ['who', 'embarked'])

features_dataset = titanic_updated.drop(['survived'], axis=1)
features = features_dataset.columns.values.tolist()
print("features: ", features)

predicted, error = run_Logistic_Regression(titanic_updated[features].as_matrix(), 
                                           titanic_updated[labels].as_matrix())

features:  ['pclass', 'age', 'sibsp', 'parch', 'fare', 'alone', 'who_1', 'who_2', 'embarked_1', 'embarked_2']
Splitting dataset...
Training set size:  (712, 10)
Test set size:  (179, 10)
Training starts...
Training ends...
Predicting...
--------------------------------------------------------------------------------
| Number of misclassifications: 35 (out of 179)
| Error:  0.19553072625698323
--------------------------------------------------------------------------------


**Observation:** no effect on the accuracy

lets try OneHot features for  `embarked`

In [13]:
titanic_updated = add_one_hot_feature(titanic, ['who', 'embarked'])

features_dataset = titanic_updated.drop(['survived'], axis=1)
features = features_dataset.columns.values.tolist()
print("features: ", features)

predicted, error = run_Logistic_Regression(titanic_updated[features].as_matrix(), 
                                           titanic_updated[labels].as_matrix())

features:  ['pclass', 'age', 'sibsp', 'parch', 'fare', 'alone', 'who_1', 'who_2', 'embarked_1', 'embarked_2']
Splitting dataset...
Training set size:  (712, 10)
Test set size:  (179, 10)
Training starts...
Training ends...
Predicting...
--------------------------------------------------------------------------------
| Number of misclassifications: 35 (out of 179)
| Error:  0.19553072625698323
--------------------------------------------------------------------------------


**Observation:** no effect

lets try OnHot feature for `sibsp`

In [14]:
titanic_updated = add_one_hot_feature(titanic, ['who', 'sibsp'])

features_dataset = titanic_updated.drop(['survived'], axis=1)
features = features_dataset.columns.values.tolist()
print("features: ", features)

predicted, error = run_Logistic_Regression(titanic_updated[features].as_matrix(), 
                                           titanic_updated[labels].as_matrix())

features:  ['pclass', 'age', 'parch', 'fare', 'embarked', 'alone', 'who_1', 'who_2', 'sibsp_1', 'sibsp_2', 'sibsp_3', 'sibsp_4', 'sibsp_5', 'sibsp_8']
Splitting dataset...
Training set size:  (712, 14)
Test set size:  (179, 14)
Training starts...
Training ends...
Predicting...
--------------------------------------------------------------------------------
| Number of misclassifications: 33 (out of 179)
| Error:  0.18435754189944134
--------------------------------------------------------------------------------


**Observation:** further improvement

lets try OneHot feature for `parch`

In [15]:
titanic_updated = add_one_hot_feature(titanic, ['who', 'sibsp', 'parch'])

features_dataset = titanic_updated.drop(['survived'], axis=1)
features = features_dataset.columns.values.tolist()
print("features: ", features)

predicted, error = run_Logistic_Regression(titanic_updated[features].as_matrix(), 
                                           titanic_updated[labels].as_matrix())

features:  ['pclass', 'age', 'fare', 'embarked', 'alone', 'who_1', 'who_2', 'sibsp_1', 'sibsp_2', 'sibsp_3', 'sibsp_4', 'sibsp_5', 'sibsp_8', 'parch_1', 'parch_2', 'parch_3', 'parch_4', 'parch_5', 'parch_6']
Splitting dataset...
Training set size:  (712, 19)
Test set size:  (179, 19)
Training starts...
Training ends...
Predicting...
--------------------------------------------------------------------------------
| Number of misclassifications: 35 (out of 179)
| Error:  0.19553072625698323
--------------------------------------------------------------------------------


**Observation:** regressed

lets try all OneHot features together

In [16]:
titanic_updated = add_one_hot_feature(titanic, ['who', 'pclass', 'embarked', 'sibsp', 'parch'])

features_dataset = titanic_updated.drop(['survived'], axis=1)
features = features_dataset.columns.values.tolist()
print("features: ", features)

predicted, error = run_Logistic_Regression(titanic_updated[features].as_matrix(), 
                                           titanic_updated[labels].as_matrix())

features:  ['age', 'fare', 'alone', 'who_1', 'who_2', 'pclass_2', 'pclass_3', 'embarked_1', 'embarked_2', 'sibsp_1', 'sibsp_2', 'sibsp_3', 'sibsp_4', 'sibsp_5', 'sibsp_8', 'parch_1', 'parch_2', 'parch_3', 'parch_4', 'parch_5', 'parch_6']
Splitting dataset...
Training set size:  (712, 21)
Test set size:  (179, 21)
Training starts...
Training ends...
Predicting...
--------------------------------------------------------------------------------
| Number of misclassifications: 34 (out of 179)
| Error:  0.18994413407821228
--------------------------------------------------------------------------------


**Observation:** The best accuracy we have got is by making a OneHot feature of `who` & `sibsp`

In [17]:
titanic_encoded = add_one_hot_feature(titanic, ['who', 'sibsp'])
titanic_encoded.head()

,survived,pclass,age,parch,fare,embarked,alone,who_1,who_2,sibsp_1,sibsp_2,sibsp_3,sibsp_4,sibsp_5,sibsp_8
0,0,3,22.0,0,7.2500,2,0,1,0,1,0,0,0,0,0
1,1,1,38.0,0,71.2833,0,0,0,1,1,0,0,0,0,0
2,1,3,26.0,0,7.9250,2,1,0,1,0,0,0,0,0,0
3,1,1,35.0,0,53.1000,2,0,0,1,1,0,0,0,0,0
4,0,3,35.0,0,8.0500,2,1,1,0,0,0,0,0,0,0


### Improve accuracy – 2) try normalizing data

In [18]:
import sklearn.preprocessing as preprocessing

for feature in ['age', 'fare']:
  titanic_encoded[feature] = pd.DataFrame(preprocessing.MinMaxScaler((0, 1)).fit_transform(titanic_encoded[[feature]]))
  
titanic_encoded.head()

,survived,pclass,age,parch,fare,embarked,alone,who_1,who_2,sibsp_1,sibsp_2,sibsp_3,sibsp_4,sibsp_5,sibsp_8
0,0,3,0.271174,0,0.014151,2,0,1,0,1,0,0,0,0,0
1,1,1,0.472229,0,0.139136,0,0,0,1,1,0,0,0,0,0
2,1,3,0.321438,0,0.015469,2,1,0,1,0,0,0,0,0,0
3,1,1,0.434531,0,0.103644,2,0,0,1,1,0,0,0,0,0
4,0,3,0.434531,0,0.015713,2,1,1,0,0,0,0,0,0,0


In [19]:
features_dataset = titanic_encoded.drop(['survived'], axis=1)
features = features_dataset.columns.values.tolist()
print("features: ", features)

predicted, error = run_Logistic_Regression(titanic_encoded[features].as_matrix(), 
                                           titanic_encoded[labels].as_matrix())

features:  ['pclass', 'age', 'parch', 'fare', 'embarked', 'alone', 'who_1', 'who_2', 'sibsp_1', 'sibsp_2', 'sibsp_3', 'sibsp_4', 'sibsp_5', 'sibsp_8']
Splitting dataset...
Training set size:  (712, 14)
Test set size:  (179, 14)
Training starts...
Training ends...
Predicting...
--------------------------------------------------------------------------------
| Number of misclassifications: 34 (out of 179)
| Error:  0.18994413407821228
--------------------------------------------------------------------------------


**Observation:** regressed, normalization didn't turn out to be helpful, discard it

In [20]:
titanic_encoded = add_one_hot_feature(titanic, ['who', 'sibsp'])
titanic_encoded.head()

,survived,pclass,age,parch,fare,embarked,alone,who_1,who_2,sibsp_1,sibsp_2,sibsp_3,sibsp_4,sibsp_5,sibsp_8
0,0,3,22.0,0,7.2500,2,0,1,0,1,0,0,0,0,0
1,1,1,38.0,0,71.2833,0,0,0,1,1,0,0,0,0,0
2,1,3,26.0,0,7.9250,2,1,0,1,0,0,0,0,0,0
3,1,1,35.0,0,53.1000,2,0,0,1,1,0,0,0,0,0
4,0,3,35.0,0,8.0500,2,1,1,0,0,0,0,0,0,0


### Improve accuracy – 3) Feature crosses

try $pclass \times embarked$

In [0]:
def add_feature_cross(data, feature1, feature2):
  new_data = data.copy()
  new_data['%s_%s'%(feature1, feature2)] = new_data[feature1]*new_data[feature2]
  return new_data

In [22]:
titanic_updated = add_feature_cross(titanic_encoded, 'pclass', 'embarked')
titanic_updated.head()

,survived,pclass,age,parch,fare,embarked,alone,who_1,who_2,sibsp_1,sibsp_2,sibsp_3,sibsp_4,sibsp_5,sibsp_8,pclass_embarked
0,0,3,22.0,0,7.2500,2,0,1,0,1,0,0,0,0,0,6
1,1,1,38.0,0,71.2833,0,0,0,1,1,0,0,0,0,0,0
2,1,3,26.0,0,7.9250,2,1,0,1,0,0,0,0,0,0,6
3,1,1,35.0,0,53.1000,2,0,0,1,1,0,0,0,0,0,2
4,0,3,35.0,0,8.0500,2,1,1,0,0,0,0,0,0,0,6


In [23]:
features_dataset = titanic_updated.drop(['survived'], axis=1)
features = features_dataset.columns.values.tolist()
print("features: ", features)

predicted, error = run_Logistic_Regression(titanic_updated[features].as_matrix(), 
                                           titanic_updated[labels].as_matrix())

features:  ['pclass', 'age', 'parch', 'fare', 'embarked', 'alone', 'who_1', 'who_2', 'sibsp_1', 'sibsp_2', 'sibsp_3', 'sibsp_4', 'sibsp_5', 'sibsp_8', 'pclass_embarked']
Splitting dataset...
Training set size:  (712, 15)
Test set size:  (179, 15)
Training starts...
Training ends...
Predicting...
--------------------------------------------------------------------------------
| Number of misclassifications: 34 (out of 179)
| Error:  0.18994413407821228
--------------------------------------------------------------------------------


**Observation:** model regressed

try $age \times embarked$

In [24]:
titanic_updated = add_feature_cross(titanic_encoded, 'age', 'embarked')

features_dataset = titanic_updated.drop(['survived'], axis=1)
features = features_dataset.columns.values.tolist()
print("features: ", features)

predicted, error = run_Logistic_Regression(titanic_updated[features].as_matrix(), 
                                           titanic_updated[labels].as_matrix())

features:  ['pclass', 'age', 'parch', 'fare', 'embarked', 'alone', 'who_1', 'who_2', 'sibsp_1', 'sibsp_2', 'sibsp_3', 'sibsp_4', 'sibsp_5', 'sibsp_8', 'age_embarked']
Splitting dataset...
Training set size:  (712, 15)
Test set size:  (179, 15)
Training starts...
Training ends...
Predicting...
--------------------------------------------------------------------------------
| Number of misclassifications: 33 (out of 179)
| Error:  0.18435754189944134
--------------------------------------------------------------------------------


**Observation:** no effect

try $age \times pclass$

In [25]:
titanic_updated = add_feature_cross(titanic_encoded, 'age', 'pclass')

features_dataset = titanic_updated.drop(['survived'], axis=1)
features = features_dataset.columns.values.tolist()
print("features: ", features)

predicted, error = run_Logistic_Regression(titanic_updated[features].as_matrix(), 
                                           titanic_updated[labels].as_matrix())

features:  ['pclass', 'age', 'parch', 'fare', 'embarked', 'alone', 'who_1', 'who_2', 'sibsp_1', 'sibsp_2', 'sibsp_3', 'sibsp_4', 'sibsp_5', 'sibsp_8', 'age_pclass']
Splitting dataset...
Training set size:  (712, 15)
Test set size:  (179, 15)
Training starts...
Training ends...
Predicting...
--------------------------------------------------------------------------------
| Number of misclassifications: 34 (out of 179)
| Error:  0.18994413407821228
--------------------------------------------------------------------------------


**Observation:** model regressed

### Final observation:
We were able to improve the model accuracy a little by using OneHot features, but Feature Crosses didn't turn out be that useful in the given permutations that were tried.